## **Using Pre-trained Embeddings for Text Classification**

We have studied different techniques of generating word embeddings -- high dimensional vectorial representations that are intended to aggregate the different contexts in which words appear in text. This is an unsupervised task that needs a lot of data to accurately generate word representations. Hence word embeddings are often pretrained. Note that pre-trained embeddings are **static representations** that do not alter for the words depending on the context in which they appear.

Let us see how we can use these pre-trained embeddings for a popular NLP task like Sequence Classification. This task entails viewing text as a sequence of tokens and generating a single label for the entire sequence for a downstream classification task. In this notebook we will choose ***Text Classification*** as a demonstration of this task.

We want to use the [AG News dataset](https://paperswithcode.com/sota/text-classification-on-ag-news), which is a corpus of news articles constructed by assembling titles and description fields of articles from the 4 largest classes: “World”, “Sports”, “Business”, “Sci/Tech”.

#### **0. Housekeeping Steps**

Let us perform the necessary housekeeping steps before procedding further with the Machine Learning task at hand. We probably need to install some packages before we can import them.

In [13]:
!pip install -U datasets

In [14]:
!pip uninstall torch torchtext -y
!pip cache purge
!pip install torch==2.3.0
!pip install torchtext==0.18

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchtext 0.18.0
Uninstalling torchtext-0.18.0:
  Successfully uninstalled torchtext-0.18.0
Files removed: 78
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.0 MB/s eta 0:00:00


We need to link our Colab notebook to our Google Drive so we can both save and load our necessary models and data. To do this we need to mount our Google Drive locally. Please be mindful of repeating this step.

In [15]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [16]:
#sanity check to ensure drive was properly mounted
!ls /content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks

 Day1-rough.ipynb
 e2e-dataset
'Homework1: Text Classification using Pre-trained Embeddings.ipynb'
'Homework2: Text Classification using Finetuning.ipynb'
'Lab 1+2: PyTorch+Neural-Networks-for-Beginners.ipynb'
'Lab 3: Classification for NLP Demo.ipynb'
'Lab 4: Hugging_Face_Transformers_Tutorial'
'Lab 5: Token Classification with HuggingFace Transformers'
'Overview of Colaboratory Features'
 python-dictionary-review.ipynb
 sample_hf_trainer
 sample_pte_trainer
'Solved-Homework1: Text Classification using Pre-trained Embeddings.ipynb'
'Solved-Homework2: Text Classification using Finetuning.ipynb'
 sst-model


NOTE: In order to load and save models we need to set the model path. First, create a folder named `sample_pte_model` in this path where you can save your best performing model.

In [22]:
model_path = "/content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks/sample_pte_trainer/"

#### **1. Imports**

Let us start with all the necessary imports

In [1]:
from collections import defaultdict, Counter
import json
import numpy as np

from matplotlib import pyplot as plt

import torch
import torchtext
from torchtext.data import get_tokenizer
from torchtext.vocab import vocab
from datasets import load_dataset, DatasetDict
from torch.utils.data import DataLoader

/usr/local/lib/python3.11/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.11/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.11/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated

In [2]:
import torch.nn as nn
import torch.optim as optim

In [3]:
tokenizer = get_tokenizer("basic_english")
tokens = tokenizer("You can now install TorchText using pip!")
tokens

['you', 'can', 'now', 'install', 'torchtext', 'using', 'pip', '!']

#### **2. Loading the data**

We load the dataset from the `torchtext.datasets` package. These datasets are already split into training and test.

In [4]:
print("Loading dataset...")
dataset_name = "ag_news"
ag_news_dataset = load_dataset(dataset_name)
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

Loading dataset...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [5]:
ag_news_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

#### **3. Building a Vocabulary**

Note that with static embeddings, we need to build a vocabulary as well to go alongside it.
We can think of this as a dictionary like structure, where keys are words in our vocabulary and embeddings are values we retrieve from the dicttionary when needed.

The vocabulary is built out of every word we encounter in the training data. Of course, we can build a vocabulary of ngrams as well instead of words -- the `torchtext` package supports this by taking the n-gram size we can to extract from the text as an argument. But let's keep things simple here.

The `torchtext` vocabulary object takes in as arguments a `Counter` like iterator and a `min_freq` that acts as a minimum threshold frequency that needs to be met by a word to be considered to be part of the vocabulary.

Counter is an unordered collection where elements are stored as `Dict` keys and their count as dict value. For reference, a `Counter` object is demonstrated here:

```python
# Counter with initial values
counter = Counter(['a', 'a', 'b'])
print(counter)  # Counter({'a': 2, 'b': 1})
```

In [6]:
print('Building vocab...')
counter = Counter()
for record in ag_news_dataset['train']:
    text = record['text']
    counter.update(torchtext.data.utils.ngrams_iterator(tokenizer(text),ngrams=1))

Building vocab...


In [7]:
# vocabulary = vocab(counter, min_freq=1)
min_freq=1
vocabulary = {x: count for x, count in counter.items() if count >= min_freq}
print(f'Vocabulary size: {len(vocabulary)}')

Vocabulary size: 95810


In [8]:
# We are just converting our vocabularly to a list to be able to index into it
# Sorting is not necessary, we sort to show an ordered word_to_ind dictionary
# That being said, we will see that having the index for the padding token
# be 0 is convenient as some PyTorch functions use it as a default value
# such as nn.utils.rnn.pad_sequence, which we will cover in a bit
ix_to_word = ["<pad>","<unk>"]+sorted(list(vocabulary.keys()))

# Add the unknown token to our vocabulary
vocabulary["<pad>"]=1
vocabulary["<unk>"]=1
print(f'Vocabulary size: {len(vocabulary)}')

# Creating a dictionary to find the index of a given word
word_to_ix = {word: ind for ind, word in enumerate(ix_to_word)}
len(word_to_ix)

Vocabulary size: 95812


95812

In [9]:
print(ix_to_word[0])
print(ix_to_word[1])

<pad>
<unk>


Now we are ready to convert sentences into a sequence of indices corresponding to each token.

In [10]:
# Given a sentence of tokens, return the corresponding indices
def convert_token_to_indices(sentence, word_to_ix):
  indices = []
  for token in sentence:
    # Check if the token is in our vocabularly. If it is, get it's index.
    # If not, get the index for the unknown token.
    if token in word_to_ix:
      index = word_to_ix[token]
    else:
      index = word_to_ix["<unk>"]
    indices.append(index)
  return indices


# Show an example
example_sentence = ["we", "always", "come", "to", "tanganyka"]
example_indices = convert_token_to_indices(example_sentence, word_to_ix)
restored_example = [ix_to_word[ind] for ind in example_indices]

print(f"Original sentence is: {example_sentence}")
print(f"Going from words to indices: {example_indices}")
print(f"Going from indices to words: {restored_example}")

Original sentence is: ['we', 'always', 'come', 'to', 'tanganyka']
Going from words to indices: [92705, 11856, 24029, 86679, 1]
Going from indices to words: ['we', 'always', 'come', 'to', '<unk>']


#### **4. Dataset Preprocessing**

Next we tokenize and prepare the data. As you might remember, the tokenizer executes the following steps:

1. Split text into tokens and convert them into word ids
2. Padding the text so all inputs are of the same length
3. Apply truncation when needed by setting a max length for sequences.
4. Convert sentences into batches

In [11]:
from torch.utils.data import DataLoader
from functools import partial

def custom_collate_fn(batch, word_to_ix):
  # Break our batch into the training examples (x) and labels (y)
  # We are turning our x and y into tensors because nn.utils.rnn.pad_sequence
  # method expects tensors. This is also useful since our model will be
  # expecting tensor inputs.

  x = [record['text'] for record in batch]
  y = torch.LongTensor([record['label'] for record in batch])

  # Convert the train examples into indices.
  x = [convert_token_to_indices(s, word_to_ix) for s in x]

  # We will now pad the examples so that the lengths of all the example in
  # one batch are the same, making it possible to do matrix operations.
  # We set the batch_first parameter to True so that the returned matrix has
  # the batch as the first dimension.
  pad_token_ix = word_to_ix["<pad>"]

  # pad_sequence function expects the input to be a tensor, so we turn x into one
  x = [torch.LongTensor(x_i) for x_i in x]
  x_padded = nn.utils.rnn.pad_sequence(x, batch_first=True, padding_value=pad_token_ix)

  # We are now ready to return our variables. The order we return our variables
  # here will match the order we read them in our training loop.
  return x_padded, y

Now, we see the DataLoader in action. Because our version of the collate_fn function will need to access to our `word_to_ix` dictionary (so that it can turn words into indices), we will make use of the `partial` function in Python, which passes the parameters we give to the function we pass it.

In [12]:
# Parameters to be passed to the DataLoader
batch_size = 16
shuffle = True
collate_fn = partial(custom_collate_fn, word_to_ix=word_to_ix)

In [13]:
train_dataloader = DataLoader(ag_news_dataset['train'], batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)
eval_dataloader = DataLoader(ag_news_dataset['test'], batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)
count=0
for batch_i, batch in enumerate(eval_dataloader):
    if(count<2):
        print(batch)
        count+=1
    else:
        break

(tensor([[    1, 31226, 50458,  ...,     0,     0,     0],
        [    1, 44644, 53408,  ...,     0,     0,     0],
        [    1, 20244, 44644,  ...,     0,     0,     0],
        ...,
        [    1, 58625, 90778,  ...,     0,     0,     0],
        [    1, 83119,     1,  ...,     0,     0,     0],
        [    1,  8962, 94977,  ...,     0,     0,     0]]), tensor([1, 0, 3, 1, 3, 2, 2, 1, 0, 1, 2, 2, 0, 0, 2, 1]))
(tensor([[    1, 31226, 61141,  ...,     0,     0,     0],
        [    1, 31226, 83119,  ...,     0,     0,     0],
        [    1, 31226, 92030,  ...,     0,     0,     0],
        ...,
        [    1, 44644, 58625,  ...,     0,     0,     0],
        [    1,   664,     1,  ...,     0,     0,     0],
        [    1, 61141, 69487,  ...,     0,     0,     0]]), tensor([2, 1, 2, 2, 0, 2, 1, 3, 0, 1, 1, 3, 3, 3, 0, 2]))


#### **5. Using a Pre-Trained Embedding**

Here we will use Glove embeddings to pre-populate the matrix for our embedding layer in the neural network that we will use and not train the embeddings any further.

GloVe object has 2 parameters: name and dim (accepted values: 25, 50, 100 and 200).



In [14]:
from torchtext.vocab import GloVe
glove_embeddings = GloVe(name='6B', dim=100)

.vector_cache/glove.6B.zip: 862MB [02:38, 5.43MB/s]                           
100%|█████████▉| 399999/400000 [00:19<00:00, 20578.36it/s]


In [15]:
word = 'basic'
torch.tensor(glove_embeddings.vectors[glove_embeddings.stoi[word]])

/tmp/ipython-input-15-3155615760.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(glove_embeddings.vectors[glove_embeddings.stoi[word]])


tensor([-0.2686,  0.6559,  0.0647,  0.4969, -0.0171, -0.2971, -0.2498, -0.1062,
        -0.0184,  0.5199,  0.1508, -0.2935,  0.2602,  0.2436, -0.1617, -0.3219,
        -0.0390,  0.4198, -0.0820, -0.2439, -0.0368, -0.7335,  0.7090, -0.3332,
         0.1184, -0.5976, -0.1186, -0.7950, -0.9640,  0.0213, -0.2473,  0.8511,
        -0.3918, -0.3747, -0.0431,  0.5516, -0.0717,  0.4992,  0.0496, -0.2164,
        -0.0041, -0.0680,  0.4032,  0.0912, -0.4820,  0.4335, -0.7191, -0.1180,
        -0.4709, -0.6470,  0.3206,  0.1386, -0.1771,  0.6697,  0.2996, -1.1808,
         1.2482, -0.4098,  2.3297, -0.0245,  0.1935, -0.0345, -0.6899, -0.4895,
         0.9623,  0.1639, -0.3837, -0.3447,  0.1348, -0.0032, -0.2865,  0.3355,
         0.4169,  0.2386,  0.4150,  0.8600, -0.4276,  0.3356, -0.6559, -0.3183,
         0.2698,  0.0892, -0.4922,  0.0778, -1.6511,  1.0244, -0.0798, -0.3221,
        -0.1591,  0.5024,  0.0442, -0.1181, -0.2758,  0.3666, -0.6952, -0.3677,
        -0.2520, -1.4229,  0.7617,  0.78

Loaded vocabulary has the following basic operations:

*   vocab.stoi dictionary allows us to convert word into its dictionary index
*   vocab.itos does the opposite - converts number into word
*   vocab.vectors is the array of embedding vectors, so to get the embedding of a word s we need to use `vocab.vectors[vocab.stoi[s]]`





#### **6. Model Construction**

In [16]:
class PTEmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        # self.embedding.weight.requires_grad = False
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

#### **7. Training and Validation**

Initialize model.

Now that we have downloaded our pre-trained embeddings, let us prepare our embedding matrix that can be fed directly into our neural network.
Note that we need to take into account that vocabularies of pre-trained embedding and our text corpus will likely not match so we will initialize weights for the missing words with random values.

In [19]:
#set seed and device
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#initialize model
vocab_size = len(word_to_ix) # our vocabulary size
embed_size = len(glove_embeddings.vectors[0])
model = PTEmbedClassifier(vocab_size,embed_size,len(classes)).to(device)

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for word, index in word_to_ix.items():
    # print(word, index)
    try:
        model.embedding.weight[index].data = torch.tensor(glove_embeddings.vectors[glove_embeddings.stoi[word]])
        found+=1
    except:
        model.embedding.weight[index].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")


Populating matrix, this will take some time...

/tmp/ipython-input-19-1428369944.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  model.embedding.weight[index].data = torch.tensor(glove_embeddings.vectors[glove_embeddings.stoi[word]])


Done, found 62487 words, 33325 words missing


Next we write the training and validation loop

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
#define a loss function
loss = torch.nn.CrossEntropyLoss().to(device)

#define an optimizer
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

#number of epochs
num_epochs = 100

best_val_loss = float("inf")
save_model_dict = {}
num_training_steps = len(train_dataloader)
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    # training
    model.train()
    training_losses = []
    for batch_i, batch in enumerate(train_dataloader):

        optimizer.zero_grad()

        # copy input to device
        text = batch[0].to(device)
        labels = batch[1].to(device)

        output = model(text)
        # matching shapes for sanity check
        # print('::', output.shape, labels.shape)
        training_loss = loss(output,labels)
        training_losses.append(training_loss.item())

        #backprop and update params by taking an optimization step
        training_loss.backward()
        optimizer.step()
        progress_bar.update(1)
    print("Mean Training Loss", np.mean(training_losses))

    # validation
    val_loss = 0
    model.eval() #important to call because we dont want to collect gradients
    for batch_i, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            # copy input to device
            text = batch[0].to(device)
            labels = batch[1].to(device)

            output = model(text)
        val_loss += loss(output,labels)

    avg_val_loss = val_loss / len(eval_dataloader)
    print(f"Validation loss: {avg_val_loss}")
    if avg_val_loss < best_val_loss:
        print("Saving checkpoint!")
        best_val_loss = avg_val_loss
        save_model_dict = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': best_val_loss,
        }
        print(save_model_dict.keys())
    print()

torch.save({
    'epoch': save_model_dict['epoch'],
    'model_state_dict': save_model_dict['model_state_dict'],
    # 'optimizer_state_dict': optimizer.state_dict(),
    'val_loss': save_model_dict['val_loss'],
    },
    f"{model_path}epoch_{save_model_dict['epoch']}.pt"
)

#### **8. Evaluate your model on Test Data**
Now we use our finetuned model to evaluate the test set. We use performance metrics from sklearn.metrics to test the effectiveness of our model on unseen test data.

In order to do that, run the finetuned model you have just saved on your test data and report the following performance metrics:

*   Accuracy
*   F1 Score


In [20]:
from sklearn.metrics import accuracy_score

In [23]:
#load validation data in one batch
model_path = "/content/gdrive/MyDrive/DSSI-2024-Week2-internal/notebooks/sample_pte_trainer/"
print(f"new batch size = {len(ag_news_dataset['test'])}")
eval_dataloader = DataLoader(ag_news_dataset['test'], batch_size=len(ag_news_dataset['test']), shuffle=shuffle, collate_fn=collate_fn)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#initialize model
model = PTEmbedClassifier(vocab_size,embed_size,len(classes)).to(device)

#load model weights
saved_model_path = f"{model_path}epoch_85.pt"
checkpoint = torch.load(saved_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
epoch = checkpoint['epoch']
val_loss = checkpoint['val_loss']

#sanity check
print(f"epoch:{epoch} val_loss:{val_loss}")

new batch size = 7600
epoch:85 val_loss:1.1195861101150513


In [24]:
model.eval()
test_batch_preds = []
test_batch_true = []
for batch_i, batch in enumerate(eval_dataloader):
    with torch.no_grad():
        # copy input to device
        text = batch[0].to(device)
        labels = batch[1].to(device)

        outputs = model(text)
        print(torch.max(outputs, dim=1))
        _, predicted_labels = torch.max(outputs, dim=1)
        print(predicted_labels)

        test_batch_preds.append(predicted_labels)
        test_batch_true.append(labels)

torch.return_types.max(
values=tensor([1.1263, 0.9015, 1.7295,  ..., 0.8061, 1.8491, 1.7713], device='cuda:0'),
indices=tensor([2, 2, 3,  ..., 1, 3, 3], device='cuda:0'))
tensor([2, 2, 3,  ..., 1, 3, 3], device='cuda:0')


In [25]:
print(len(test_batch_preds),len(eval_dataloader))
y_pred = torch.cat(test_batch_preds, dim=0)
y_true = torch.cat(test_batch_true, dim=0)
#sanity check -> dimension 0 of your logits tensor should be same as the size of the test dataset
print(y_pred.shape,len(ag_news_dataset['test']),y_true.shape)

1 1
torch.Size([7600]) 7600 torch.Size([7600])


In [26]:
#Convert the logits to predicted labels
y_pred = y_pred.cpu().numpy()
y_true = y_true.cpu().numpy()
print(y_true[:10])
print(y_pred[:10])

#sanity check: should have as many predictions as labels
assert len(y_pred)==len(y_true)

[2 2 3 0 1 2 0 2 3 0]
[2 2 3 3 3 1 1 3 3 1]


In [27]:
print('Accuracy Score:',accuracy_score(y_true, y_pred))

Accuracy Score: 0.3717105263157895
